#### [PREV](3.OOP_Polynomial_Liner_Regression.ipynb) | [HOME](../README.md) | [NEXT](5.OOP_KNN.ipynb)

---

# OOP Logistic Regression Implementation

This Jupyter Notebook provides a basic implementation of Logistic Regression for classification. The data set contains measurements on a single species of insect captured on two continents. We want to predict which continent (0 or 1) an insect was captured on given the other measurements we have (wing size, latitude and sex).

#### Step 1: Import Dependencies

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error

#### Step 2: Import And Visualise The Data

In [ ]:
# Import data as a Pandas DataFrame
insects = pd.read_csv('./data/insects.csv', sep='\t')
insects = pd.DataFrame({
    'continent': insects['continent'],
    'latitude': insects['latitude'],
    'sex': insects['sex'],
    'wingsize': insects['wingsize']
})

# Features variable
X_insects = insects[['wingsize', 'latitude', 'sex']]
# Target variable
y_insects = insects['continent']

insects.head()

In [ ]:
# Plot the data, colour coded by sex
plt.scatter(insects.wingsize, insects.continent, c=insects['sex'])
plt.xlabel("Wing size")
plt.ylabel("Continent")
plt.colorbar(label="Sex")
plt.show()

#### Step 3: Instantiate a LogisticRegression Object

In [ ]:
# max_iter is increased so the solver has enough iterations to find the best fit (converge)
insects_regression = LogisticRegression(max_iter=1000)

#### Step 4: Fit the model

In [ ]:
insects_regression.fit(X_insects, y_insects)

Now that the model has been fit, we can use the `predict` method to predict which continent (0 or 1) the model thinks an insect was captured on.

In [ ]:
# Predict the target for the whole dataset
insects_predictions = insects_regression.predict(X_insects)

# Predict the target for a new data point
new_insect = pd.DataFrame({
    'wingsize': [900],
    'latitude': [40],
    'sex': [0]
})
new_insect['continent'] = insects_regression.predict(new_insect)
print(f"New insect is:\n{new_insect}")

Additionally (and in many cases more usefully), the `LogisticRegression` has a `predict_proba` method that produces a predicted **probability**.

In [ ]:
# Predict the probabilities for the whole dataset
insects_probabilities = insects_regression.predict_proba(X_insects)

# Predict the probabilities for a new data point
new_insect = pd.DataFrame({
    'wingsize': [900],
    'latitude': [40],
    'sex': [0]
})
new_insect_probabilities = insects_regression.predict_proba(new_insect)
print(f"Probabilities for new insect [continent 0, continent 1]:\n{new_insect_probabilities}")

#### Step 5: Plot the Predictions

In [ ]:
# Plot the predictions compared to the actual data
fig, axs = plt.subplots(1, 3, figsize=(14, 5))

axs[0].scatter(insects.wingsize, insects.continent, s=40, c=insects['sex'])
axs[0].set_title("Actual Data")
axs[0].set_xlabel("Wing size")
axs[0].set_ylabel("Continent")

axs[1].scatter(insects.wingsize, insects_probabilities[:, 1], s=40, c=insects['sex'])
axs[1].set_title("Prediction Probabilities")
axs[1].set_xlabel("Wing size")
axs[1].set_ylabel("Probability of Continent 1")

axs[2].scatter(insects.wingsize, insects_predictions, s=40, c=insects['sex'])
axs[2].set_title("Predicted Data")
axs[2].set_xlabel("Wing size")
axs[2].set_ylabel("Continent Prediction")

plt.tight_layout()
plt.show()

#### Step 6: Evaluate The Model

In [ ]:
# Get evaluative data from the model
accuracy = accuracy_score(y_true=y_insects, y_pred=insects_predictions)
print(f"Accuracy: {accuracy:.3f}")
print("Confusion matrix (rows = actual continent, columns = predicted continent):")
print(confusion_matrix(y_true=y_insects, y_pred=insects_predictions))

# MAE and MSE compare the predicted probabilities with the actual continent (lower is better)
mae = mean_absolute_error(y_true=y_insects, y_pred=insects_probabilities[:, 1])
mse = mean_squared_error(y_true=y_insects, y_pred=insects_probabilities[:, 1])
print("MAE:", mae)
print("MSE:", mse)

In [ ]:
# Optional: Manually evaluate the loss and cost of the model
model_loss = pd.DataFrame({
    'Target': y_insects,
    'Predicted result': insects_predictions,
    'Loss': (insects_predictions != y_insects).astype(int)  # 1 = wrong prediction, 0 = correct
})
# Cost = the fraction of predictions that were wrong (the error rate)
model_cost = model_loss['Loss'].mean()

print(f"The cost (error rate) of this model is {model_cost:.5f}")
print(f"The model predicted incorrectly {model_loss['Loss'].sum()} times out of {model_loss.shape[0]} predictions")
print(model_loss)